In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [12]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,legacyLink,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,url,indicator
0,6755399447111241,2025-05-14T17:44:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-19T12:25:16Z,1.0,4,FBI Email Alert May 14 2025,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.236.176.16
1,5265101,2025-01-23T19:51:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-19T12:12:28Z,1.0,6,CISA conducted a hunt on IoC's obtained from o...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 546888, 'dateAdded': '2025-01-23T19:51...",NaN,NaN,NaN,NaN,NaN,NaN,146.190.241.65
2,5265088,2025-01-23T19:51:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-19T12:09:04Z,3.0,57,CISA conducted a hunt on IoC's obtained from o...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 546888, 'dateAdded': '2025-01-23T19:51...",NaN,NaN,NaN,NaN,NaN,NaN,138.197.101.95
3,6755399459033722,2025-06-16T17:42:11Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-19T12:08:55Z,3.0,71,RITM0589984,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.40.4.149
4,5629499564010904,2025-08-18T13:49:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-19T12:08:50Z,3.0,80,INC9198174,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.86.35.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1287,4503719,2024-01-23T19:14:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:43Z,3.0,80,A phishing email with a .htm file containing a...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 238537, 'name': 'fake voicemail', 'las...","[{'id': 308984, 'dateAdded': '2024-01-23T19:14...",NaN,NaN,NaN,https://aka.ms/LearnAboutSenderIdentification,NaN,aka.ms/learnaboutsenderidentification,aka.ms/learnaboutsenderidentification
1288,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 34822, 'name': 'business email comprom...","[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,geo.netsupportsoftware.com/location/loca.asp
1290,4585199,2024-04-30T15:02:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-05-05T23:23:15Z,4.0,54,VA users received email messages from <Good Ch...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 39680, 'name': 'Initial Access: Phishi...",NaN,NaN,NaN,NaN,NaN,NaN,track.christianityreport.net,track.christianityreport.net
1291,4585198,2024-04-30T15:02:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-04-30T15:24:35Z,4.0,51,VA users received email messages from <Good Ch...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 39680, 'name': 'Initial Access: Phishi...",NaN,NaN,NaN,NaN,NaN,NaN,christianityreport.net,christianityreport.net


In [3]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"

# Fields to return from GET /v3/indicators/{indicatorId or indicator}
detail_fields = [
    "ownerName", "type", "summary", "dateAdded", "lastModified", "active",
    "confidence", "rating", "threatAssessScore", "observations",
    "tags", "attributes", "securityLabels", "associatedGroups"
]

list_of_indicators = ['207.167.64.24'] 

# Build the keys to fetch
if "observed_src" in globals() and isinstance(observed_src, pd.DataFrame) and not observed_src.empty:
    if "id" in observed_src.columns:
        indicator_keys = (
            observed_src["id"].dropna().astype(str).str.strip().tolist()
        )
    elif "summary" in observed_src.columns:
        indicator_keys = (
            observed_src["summary"].dropna().astype(str).str.strip().tolist()
        )
    else:
        indicator_keys = list_of_indicators
else:
    indicator_keys = list_of_indicators

# De-duplicate while preserving order
seen = set()
ordered_keys = []
for k in indicator_keys:
    if k and k not in seen:
        seen.add(k)
        ordered_keys.append(k)

final_results = []

# Query indicators (single-resource endpoint only)
for key in ordered_keys:
    display(f"Querying indicator: {key}")
    try:
        # URL-encode the path segment (works for both numeric IDs and summary strings)
        path_key = urllib.parse.quote(str(key), safe="")
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{path_key}?fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            result = response.json() or {}
            final_results.append(result)
    except Exception as e:
        pass

# Normalize results
normalized_data = []
for result in final_results:
    data_item = result.get('data', {})
    # v3 single GET usually returns a dict; occasionally data['indicator'] may hold it
    if isinstance(data_item, dict):
        record = data_item.get('indicator', data_item)
        if isinstance(record, dict) and 'summary' in record:
            normalized_data.append(record)
    elif isinstance(data_item, list):
        # Very rare shape; keep parity with earlier normalizer
        for item in data_item:
            if isinstance(item, dict) and 'summary' in item:
                normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    # Derive a simple 'indicator' token from summary (e.g., first token)
    if 'summary' in observed_src.columns:
        observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()

    # Drop dupes on 'indicator' if it exists
    if 'indicator' in observed_src.columns:
        observed_src.drop_duplicates(subset='indicator', inplace=True)

    # Filter to lastObserved >= start, if lastObserved exists
    if 'lastObserved' in observed_src.columns:
        observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
        #observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying indicator: 207.167.64.24'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,summary,observations,lastObserved,privateFlag,active,activeLocked,ip,legacyLink,tags.data,indicator
0,5629499556005918,2025-06-30T17:15:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-18T07:46:16Z,3.0,73,RITM0594044,207.167.64.24,247054,2025-08-17 00:00:00+00:00,False,True,False,207.167.64.24,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 749, 'name': 'malicious', 'lastUsed': ...",207.167.64.24


In [4]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src if desired
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')

display(observed_src_with_tags[['indicator', 'tag_list']])

,indicator,tag_list
0,207.167.64.24,"[malicious, CDC Splunk API, Observed, HHS Splu..."


In [5]:
# List of tags to count (case-insensitive, strip whitespace)
tags_of_interest = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections"
]

# Normalize tag_list to lowercase for matching
def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag.lower())

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()
    # Add a column to observed_src with the list of matching "Botnet" tags per indicator
    botnet_tags = set(tags_of_interest)
    def extract_botnet_tags(tag_list):
        if not isinstance(tag_list, list):
            return []
        return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in {b.lower() for b in botnet_tags}]

    # Map indicator to its tag_list for efficient lookup
    indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

    observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))
# Display as a DataFrame for readability
pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])

,Tag,Count
0,Scanning,0
1,DDoS,0
2,Spam,0
3,Phishing,0
4,Cryptojacking,0
5,Credential Stuffing,0
6,Ransomware,0
7,Data Theft,0
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [10]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,observations,lastObserved,privateFlag,active,activeLocked,ip,legacyLink,tags.data,indicator,Botnet
0,5629499556005918,2025-06-30T17:15:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-18T07:46:16Z,3.0,73,RITM0594044,...,247054,2025-08-17 00:00:00+00:00,False,True,False,207.167.64.24,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 749, 'name': 'malicious', 'lastUsed': ...",207.167.64.24,[]


In [11]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_18948\1319479530.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250819.csv']"

'Loaded data from 1 files.'

In [12]:
import pandas as pd

df = observed_src.copy()

#df = df[df['indicator'].isin(observed_data_df['indicator'])]
# --- FULL TAGS UNNEST + PARTNERS ---

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)

# --- GROUPS VIA EXPLODE + GROUPBY ---

#groups_exploded = (
#    df[['indicator', 'associatedGroups.data']]
#      .explode('associatedGroups.data')
#      .dropna(subset=['associatedGroups.data'])
#)

#group_norm = pd.json_normalize(
#    groups_exploded['associatedGroups.data']
#).rename(columns={'id':'group_id','name':'group_name'})

#groups_exploded = groups_exploded.reset_index(drop=True).join(group_norm)

#group_agg = (
#    groups_exploded
#      .drop_duplicates(subset=['indicator','group_id','group_name'])
#      .groupby('indicator', as_index=False)
#      .agg({
#          'group_id':   lambda ids: list(ids),
#          'group_name': lambda names: list(names)
#      })
#      .rename(columns={'group_id':'group_ids','group_name':'group_names'})
#)

# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','source','address','url'
]
# Remove 'hostName', 'dnsActive', 'whoisActive' from first_cols to avoid KeyError

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c in df.columns]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
#      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# apply to all your formerly‑list columns
#for col in ['partners','group_ids','group_names'] + tag_fields:
#    agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

# Only apply to columns that exist in agg_df to avoid KeyError
for col in ['partners', 'group_ids', 'group_names'] + tag_fields:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)
    
display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,active,activeLocked,ip,legacyLink,Botnet,tag_id,tag_name,tag_lastUsed,tag_description,partners
0,207.167.64.24,5629499556005918,2025-06-30T17:15:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-18T07:46:16Z,3.0,73,...,True,False,207.167.64.24,https://hvs.threatconnect.com/auth/indicators/...,[],"749, 23575, 23576, 23577, 23586, 23630, 23769,...","malicious, CDC Splunk API, Observed, HHS Splun...","2025-08-18T14:50:29Z, 2025-08-19T10:13:24Z, 20...",Observations reported by the HHS Ofice of the ...,"CDC, HHS, IHS, NIH, CMS, FDA, OS, DHA"


In [13]:
# ──────────────────────────────────────────────────────────────────────────────
# Enrich only final filtered indicators (Shodan & VirusTotalV3) and flatten
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

indicator_values = (
    agg_df["summary"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

display(f"Enriching {len(indicator_values)} indicators with Shodan & VirusTotalV3...")

enriched_results = []
failed = []

for value in indicator_values:
    try:
        encoded_value = urllib.parse.quote(value, safe="")
        q = urllib.parse.urlencode({"type": ["Shodan", "VirusTotalV3"]}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        # Build a fresh RequestObject each loop (adjust to your SDK)
        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        resp.raise_for_status()

        data = resp.json()
        data["summary"] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({"summary": value, "error": str(e)})

# If nothing enriched, just carry on
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()

else:
    # One row per summary from enrichment payload
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset="summary", keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on="summary", how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[["summary", COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat["summary"] = exploded["summary"].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            """Flatten one level of lists in a sequence, keep dicts intact."""
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            # if everything is hashable & not dict/list, dedupe
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat  # keep as-is when dicts/lists present

        obj_cols = enrich_flat.select_dtypes("object").columns.difference(["summary"])
        num_cols = enrich_flat.columns.difference(obj_cols.union({"summary"}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        # choose your numeric aggregation; 'max' or 'first'
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby("summary", as_index=False)
              .agg(agg_dict)
        )

        # Remove raw list col and merge flattened cols back
        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset="summary")
                       .merge(enrich_wide, on="summary", how="left")
        )

        display(f"Successfully enriched and merged {len(df_enriched)} indicators.")
    else:
        display("Enrichment column not found; nothing to flatten.")

# Optional: report/log failures
if failed:
    display(f"{len(failed)} indicators failed to enrich.")
    # Example: pd.DataFrame(failed).to_json("enrich_failures.json", orient="records")


'Enriching 1 indicators with Shodan & VirusTotalV3...'

C:\Users\jaskew\AppData\Local\Temp\ipykernel_23376\2721959216.py:88: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  return list(pd.unique(flat))


'Successfully enriched and merged 1 indicators.'

In [14]:
recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')

recent_tags

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,data.activeLocked,enrich_asn,enrich_city,enrich_country,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,207.167.64.24,2025-06-30T17:15:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-18T07:46:16Z,3.0,73,RITM0594044,...,False,[AS399045],[Kansas City],[United States],"[DediOutlet, LLC]","[{'transport': 'tcp', 'port': 21, 'data': '220...",[Sullivan's Hosting LLC],[scanner],"[{'name': 'CVE-2024-38475', 'port': 80, 'descr...",21.0


In [ ]:
from pymongo import MongoClient

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Controlled_Scoring_Data"]
collection = db["controlled_enriched_observation_Data"]

# Convert DataFrame to dictionary records
records = recent_tags.to_dict(orient="records")

# Insert records into MongoDB
result = collection.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} documents into enriched_observation_Data.")

client.close()


In [97]:
from pymongo import MongoClient
import pandas as pd

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Larger_Scoring_Data"]
collection = db["larger_enriched_observation_Data"]

# Pull data back from MongoDB
docs = list(collection.find())
recent_tags = pd.DataFrame(docs)
print(f"Pulled {len(recent_tags)} documents from enriched_observation_Data.")


Pulled 10000 documents from enriched_observation_Data.


In [99]:
recent_tags = recent_tags.head(50)
recent_tags.loc[recent_tags['enrich_vtMaliciousCount'].isnull(), 'enrich_vtMaliciousCount'] = 0
display(recent_tags)

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,68a60b186a95e5f0822c49b8,0-------zpital-of-philadelphia-crowdstrike.gw....,2024-07-29T16:43:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:41Z,3.0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,68a60b186a95e5f0822c49b9,00512-okta-0cd1fe6e4d31c9b-12-okta.core.gw-ste...,2024-07-29T16:43:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:50Z,3.0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,68a60b186a95e5f0822c49ba,00512-okta-0cd1fe6e4d31c9b-12-okta.core.gw.cro...,2024-07-29T16:43:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:42Z,3.0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,68a60b186a95e5f0822c49bb,00518-okta-0cd1fe6e4d61c9b-18-okta-stepwell-cr...,2024-07-29T16:43:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:37Z,3.0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,68a60b186a95e5f0822c49bc,00d7ks32.life,2024-05-07T15:02:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-20T07:59:56Z,3.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
5,68a60b186a95e5f0822c49bd,0212top.online,2024-07-08T20:55:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:49Z,4.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
6,68a60b186a95e5f0822c49be,0212top.site,2024-07-08T20:55:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:39Z,4.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
7,68a60b186a95e5f0822c49bf,0212top.top,2024-07-08T20:55:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:54Z,4.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
8,68a60b186a95e5f0822c49c0,0212top.xyz,2024-07-08T20:55:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:36Z,4.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
9,68a60b186a95e5f0822c49c1,02B4571470D83163D103112F07F1C434,2025-07-23T16:41:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,File,2025-08-19T18:04:32Z,5.0,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [105]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=365)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250825.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250824.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250823.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250822.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250821.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250820.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250819.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250818.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250817.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250816.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250815.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250814.csv', 'Z:/HTOC/Data_Analytics/Data/Op

In [106]:
# Count how many times each indicator in recent_tags was seen in observed_data_df
indicator_counts = (
    observed_data_df['indicator']
    .value_counts()
    .rename('obs_count')
    .reset_index()
    .rename(columns={'index': 'indicator'})
)

# Merge the counts into recent_tags
recent_tags = recent_tags.merge(indicator_counts, on='indicator', how='left')
recent_tags['obs_count'] = recent_tags['obs_count'].fillna(0).astype(int)

display(recent_tags[['indicator', 'obs_count']])

,indicator,obs_count
0,0-------zpital-of-philadelphia-crowdstrike.gw....,0
1,00512-okta-0cd1fe6e4d31c9b-12-okta.core.gw-ste...,0
2,00512-okta-0cd1fe6e4d31c9b-12-okta.core.gw.cro...,0
3,00518-okta-0cd1fe6e4d61c9b-18-okta-stepwell-cr...,0
4,00d7ks32.life,0
5,0212top.online,0
6,0212top.site,0
7,0212top.top,0
8,0212top.xyz,0
9,02B4571470D83163D103112F07F1C434,0


In [ ]:
import re

# helper to flatten list-or-str into a list of strings
def flatten_hosts(hosts):
    if isinstance(hosts, list):
        return [h for h in hosts if isinstance(h, str)]
    elif isinstance(hosts, str):
        return [hosts]
    return []

# regex: 
# ^[A-Za-z]+       → first label is one “word” (letters only)
# (?:\.[^.]+){3}$  → exactly three more “.” followed by non-dot characters
four_label_word_prefix = re.compile(r'^[A-Za-z]+(?:\.[^.]+){3}$')

# Try to match on 'enrich_org' instead of 'enrich_hostNames'
if 'enrich_org' in recent_tags.columns:
    global matched_results
    exploded = recent_tags.explode('enrich_org')

    mask = exploded['enrich_org'].astype(str).str.match(four_label_word_prefix, na=False)

    matched = recent_tags.loc[exploded[mask].index.unique()]
    matched_results = matched[['indicator','enrich_org']]
else:
    matched = pd.DataFrame(columns=recent_tags.columns)
    matched_results = None


display(matched_results)


In [109]:
recent_tags

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount,obs_count
0,68a60b186a95e5f0822c49b8,0-------zpital-of-philadelphia-crowdstrike.gw....,2024-07-29T16:43:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:41Z,3.0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
1,68a60b186a95e5f0822c49b9,00512-okta-0cd1fe6e4d31c9b-12-okta.core.gw-ste...,2024-07-29T16:43:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:50Z,3.0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
2,68a60b186a95e5f0822c49ba,00512-okta-0cd1fe6e4d31c9b-12-okta.core.gw.cro...,2024-07-29T16:43:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:42Z,3.0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
3,68a60b186a95e5f0822c49bb,00518-okta-0cd1fe6e4d61c9b-18-okta-stepwell-cr...,2024-07-29T16:43:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:37Z,3.0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
4,68a60b186a95e5f0822c49bc,00d7ks32.life,2024-05-07T15:02:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-20T07:59:56Z,3.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
5,68a60b186a95e5f0822c49bd,0212top.online,2024-07-08T20:55:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:49Z,4.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
6,68a60b186a95e5f0822c49be,0212top.site,2024-07-08T20:55:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:39Z,4.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
7,68a60b186a95e5f0822c49bf,0212top.top,2024-07-08T20:55:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:54Z,4.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
8,68a60b186a95e5f0822c49c0,0212top.xyz,2024-07-08T20:55:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-19T07:45:36Z,4.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
9,68a60b186a95e5f0822c49c1,02B4571470D83163D103112F07F1C434,2025-07-23T16:41:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,File,2025-08-19T18:04:32Z,5.0,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0


In [140]:
import pandas as pd
import numpy as np

df = recent_tags.copy()

# ── Severity Binning ────────────────────────────
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

# ── Feature Weights ─────────────────────────────
Weights = {
    "MALICIOUS_WEIGHT": 0.50,
    "OBSERVATION_COUNT_WEIGHT": 0.25,
    "CONTINUITY_WEIGHT": 0.10,
    "BOTNET_FLAG_WEIGHT": -0.20, 
    "TC_RATING": 0.10,
    "TC_CONFIDENCE": 0.05
}

# ── Continuity Map ──────────────────────────────
CONTINUITY_MAP = {
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}

# Known max VT detections
VT_MAX = 94  
BOTNET_MULTIPLIER = 0.8   # penalize 20%
MAX_OBS_REALISTIC = 1000

# ── Raw weighted contributions ──────────────────
df['malicious_scaled'] = np.log1p(df['enrich_vtMaliciousCount'])
df['malicious_raw_score'] = (
    df['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']
)

df['obs_count_raw_score'] = (
    df['obs_count'] * Weights['OBSERVATION_COUNT_WEIGHT']
)

# Continuity contribution
df['continuity_val'] = df['type'].map(CONTINUITY_MAP).fillna(0)
df['continuity_raw_score'] = (
    df['continuity_val'] * Weights['CONTINUITY_WEIGHT']
)

# ── Botnet ─────────────────────────────────────
botnet_actions = {'Scanning','DDoS','Spam','Phishing','Cryptojacking','Credential Stuffing'}
col = df.get('Botnet', None)

if col is None:
    df['botnet_flag'] = 0
else:
    def is_botnet(val):
        if isinstance(val, (list, set, tuple)):
            return int(bool(set(map(str, val)) & botnet_actions))
        if isinstance(val, str):
            parts = {p.strip() for p in val.replace(',', ' ').split()}
            return int(bool(parts & botnet_actions))
        return 0
    df['botnet_flag'] = col.apply(is_botnet).astype(int)

# Analyst inputs
df['tc_raw_rating'] = (
    df['rating'] * Weights['TC_RATING']
)
df['tc_raw_confidence'] = (
    df['confidence'] * Weights['TC_CONFIDENCE']
)

# ── Combine into a total raw score ──────────────
df['raw_score'] = (
    df['malicious_raw_score'] +
    df['obs_count_raw_score'] +
    df['continuity_raw_score'] +
    df['tc_raw_rating'] +
    df['tc_raw_confidence']
)

df['raw_score'] = np.where(
    df['botnet_flag'] == 1,
    df['raw_score'] * BOTNET_MULTIPLIER,
    df['raw_score']
)

# ── Define a hybrid cap (decoupled from weights) ─────────────────────────
max_malicious = np.log1p(VT_MAX)     # theoretical max VT contribution (unweighted)
obs_95th = np.percentile(df['obs_count'], 95) if not df.empty else 0
max_obs = min(obs_95th, MAX_OBS_REALISTIC)    # realistic max obs_count (unweighted)
max_continuity = max(CONTINUITY_MAP.values()) # max continuity value (unweighted)

# apply weights only in raw_score, not here
RAW_SCORE_CAP = (
    max_malicious * Weights['MALICIOUS_WEIGHT'] +
    max_obs * Weights['OBSERVATION_COUNT_WEIGHT'] +
    max_continuity * Weights['CONTINUITY_WEIGHT'] +
    (100 * Weights['TC_RATING']) +
    (100 * Weights['TC_CONFIDENCE'])
)

# ── Normalize to 0–1000 scale ───────────────────
df['Threat_Score'] = 1000 * (df['raw_score'] / RAW_SCORE_CAP).clip(0, 1)

# ── Assign Severity bin ─────────────────────────
df['Severity'] = pd.cut(
    df['Threat_Score'],
    bins=scoring_bins,
    labels=label_bins,
    right=False
)

display(df[['indicator','type','enrich_vtMaliciousCount','obs_count',
            'malicious_raw_score','obs_count_raw_score',
            'continuity_raw_score','tc_raw_rating',
            'tc_raw_confidence','botnet_flag',
            'raw_score','Threat_Score','Severity']])


,indicator,type,enrich_vtMaliciousCount,obs_count,malicious_raw_score,obs_count_raw_score,continuity_raw_score,tc_raw_rating,tc_raw_confidence,botnet_flag,raw_score,Threat_Score,Severity
0,0-------zpital-of-philadelphia-crowdstrike.gw....,Host,0.0,0,0.000000,0.00,0.2,0.3,0.75,0,1.250000,46.207410,low
1,00512-okta-0cd1fe6e4d31c9b-12-okta.core.gw-ste...,Host,0.0,0,0.000000,0.00,0.2,0.3,0.75,0,1.250000,46.207410,low
2,00512-okta-0cd1fe6e4d31c9b-12-okta.core.gw.cro...,Host,0.0,0,0.000000,0.00,0.2,0.3,0.75,0,1.250000,46.207410,low
3,00518-okta-0cd1fe6e4d61c9b-18-okta-stepwell-cr...,Host,0.0,0,0.000000,0.00,0.2,0.3,0.75,0,1.250000,46.207410,low
4,00d7ks32.life,Host,0.0,0,0.000000,0.00,0.2,0.3,0.40,0,0.900000,33.269335,low
5,0212top.online,Host,0.0,0,0.000000,0.00,0.2,0.4,1.15,0,1.750000,64.690373,low
6,0212top.site,Host,0.0,0,0.000000,0.00,0.2,0.4,1.15,0,1.750000,64.690373,low
7,0212top.top,Host,0.0,0,0.000000,0.00,0.2,0.4,1.15,0,1.750000,64.690373,low
8,0212top.xyz,Host,0.0,0,0.000000,0.00,0.2,0.4,1.15,0,1.750000,64.690373,low
9,02B4571470D83163D103112F07F1C434,File,0.0,0,0.000000,0.00,0.0,0.5,5.00,0,5.500000,203.312602,medium
